# Presenter walkthrough — `session1_part2_gpu.ipynb`

**Solved copy, for driving the session live.** It clones the `session1-part2-solution` branch, so every cell below runs as-is with no editing; attendees run the exercise notebook, which does not. The narration lives in the deck, `presentation/session1_part2.pdf`.

Generated by `presentation/generate_walkthrough.py` — do not edit this notebook; re-run the script.


# SC26 Tutorial — Session 1, Part 2: Introduction to MLIR (GPU)

This is **Part 2 of 2**, continuing from `session1_part1_cpu.ipynb`. Same matrix-matrix product, `out[i, j] += sum_k a[i, k] * b[k, j]`, now on the GPU:

1. **A GPU kernel, read through.** `session1/matmul_gpu.mlir` is the matmul written by hand with `gpu.launch`. We walk through it, lower it to NVVM/PTX and run it on Colab's GPU. There is nothing to write in this step.
2. **`blas.matmul` lowered straight to a `gpu.launch` kernel** by a pattern you write — the one exercise in this part. Its setup code is provided; you create the launch and its reduction loop.

As in Part 1, you edit a source file and the notebook cells check it, with a complete matrix-*vector* version next to it as the reference.

**This needs a GPU runtime: Runtime -> Change runtime type -> T4 GPU.** Set that before running any cells below.

This notebook is self-contained: it downloads its own copy of the toolchain and rebuilds the `blas` plugin from scratch, since a new runtime shares no state with Part 1. Your Part 1 edits went with that runtime, so **this notebook starts from the Part 1 solution**: it clones the `session1-part1-solution` branch, where `blas.matmul` is already defined and verified. Nothing to re-type.

## Setup

**Run the cells in order, starting here.** Later cells use names defined by
earlier ones, so skipping ahead fails with `NameError` — or, for a name that
happens to match an IPython magic, something stranger. `Ctrl+F8` (run every cell above)
catches you up if you lose your place.

Downloads a prebuilt MLIR toolchain (`mlir-opt`, `mlir-tblgen`, the MLIR Python bindings, and CUDA runtime support) built specifically for this tutorial. See the [repo README](https://github.com/MaheshRavishankar/sc26_tutorial) for why this is a custom build rather than the official LLVM release or a pip package.

In [ ]:
REPO = "MaheshRavishankar/sc26_tutorial"
RELEASE_TAG = "session1-toolchain-v1"
ASSET_NAME = "mlir-tutorial-llvm.tar.gz"

In [ ]:
import os
import requests

# Set MLIR_TUTORIAL_INSTALL_DIR to iterate locally (e.g. in VS Code) against
# an already-built toolchain -- skips the download entirely if it's already
# there. Unset in Colab, so this always defaults to the normal /content path.
INSTALL_DIR = os.environ.get("MLIR_TUTORIAL_INSTALL_DIR", "/content/mlir-tutorial-llvm")

if os.path.exists(f"{INSTALL_DIR}/bin/mlir-opt"):
    print(f"Using existing toolchain at {INSTALL_DIR}")
else:
    tar_path = f"{INSTALL_DIR}.tar.gz"
    url = f"https://github.com/{REPO}/releases/download/{RELEASE_TAG}/{ASSET_NAME}"
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(tar_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)

    !mkdir -p {INSTALL_DIR}
    !tar xf {tar_path} -C {INSTALL_DIR} --strip-components=1

In [ ]:
import os, sys

os.environ["PATH"] = f"{INSTALL_DIR}/bin:" + os.environ["PATH"]
sys.path.insert(0, f"{INSTALL_DIR}/python_packages/mlir_core")

!mlir-opt --version

In [ ]:
from mlir.ir import Context, Module
from mlir.passmanager import PassManager
from mlir.execution_engine import ExecutionEngine
import mlir.runtime as rt
import ctypes
import numpy as np

print("MLIR Python bindings imported OK")

Also clone this repo -- every `.mlir` source below, and the `blas` dialect you extend in step 2, live here.

In [ ]:
# Set MLIR_TUTORIAL_REPO_DIR to iterate locally against an already-cloned
# checkout -- skips the clone entirely if it's already there.
# Part 2 clones the Part 1 solution branch, not main: it carries the
# blas.matmul definition you wrote in Part 1, so this notebook can get
# straight to its GPU lowering.
PART2_BRANCH = "session1-part2-solution"
REPO_DIR = os.environ.get("MLIR_TUTORIAL_REPO_DIR", "/content/sc26_tutorial")
if os.path.isdir(REPO_DIR):
    print(f"Using existing repo checkout at {REPO_DIR}")
else:
    # --no-single-branch: every branch's tip, still at depth 1. It costs
    # nothing measurable, and it means the solution branches are already in
    # your checkout if you want a solved file out of one.
    !git clone --depth 1 --no-single-branch --branch {PART2_BRANCH} https://github.com/{REPO} {REPO_DIR}

## Step 1 — a GPU kernel, read through

**`session1/matmul_gpu.mlir` is complete — read it; there is nothing to write in this step.** Run the cell below to open it in the editor.

It is Part 1's matmul written as a GPU kernel with `gpu.launch`:

- `gpu.launch` runs its body once on every thread of a grid of blocks. `blocks(...) in (...)` gives the number of blocks in x, y and z, and `threads(...) in (...)` the number of threads in each block. Inside the body, `%bx, %by, %bz` are the block ids and `%tx, %ty, %tz` the thread ids — CUDA's `blockIdx` and `threadIdx` — and `%blockX, %blockY, %blockZ` are the block sizes, CUDA's `blockDim`.
- Here it is one block of `(5, 4, 1)` threads, one per element of the 4 x 5 `out`: y over the 4 rows, x over the 5 columns. Thread `(%ty, %tx)` computes `out[%row, %col]`, reducing over `%k` as Part 1's loop nest did, and starting from the value already in `out` (the `+=`).
- `%row` and `%col` are the thread's global index, worked out the canonical way: block id times block size plus thread id, `%row = %by * %blockY + %ty` and `%col = %bx * %blockX + %tx`. With one block the block ids are 0, so these are just `%ty` and `%tx` — but the same code is right for any number of blocks.
- The `memref.cast` + `gpu.host_register` lines at the top make the host buffers reachable from the GPU. They are boilerplate, and the pattern you write in step 2 has them provided.

Why x on the columns: `memref`s are row-major, so `out[i, j]` and `b[k, j]` are contiguous in `j`. A warp is 32 threads with consecutive `threadIdx.x`, so with x on `j` those lanes touch adjacent floats — one memory transaction rather than one per lane.

The cells below round-trip it, lower it and run it.

In [ ]:
# Opens the file below. On Colab, files.view() puts it in the editor
# pane -- use it instead of the file browser, which refuses some file
# types. Running locally there is no google.colab, so this prints the
# path instead and you open it in your own editor.

try:
    from google.colab import files

    files.view(f"{REPO_DIR}/session1/matmul_gpu.mlir")   # read this
except ModuleNotFoundError:
    print("open in your editor:")
    print(f"    {REPO_DIR}/session1/matmul_gpu.mlir   (read this)")

First, parse and re-print the file with `mlir-opt`. No pass pipeline is given, so this just round-trips it through MLIR's parser and printer — the quickest way to catch a syntax error or a type mismatch in a file.

In [ ]:
MATMUL_GPU_FILE = f"{REPO_DIR}/session1/matmul_gpu.mlir"

!mlir-opt {MATMUL_GPU_FILE}
assert _exit_code == 0

Now parse it into an in-memory MLIR module on the Python side.

In [ ]:
ctx = Context()
module = Module.parseFile(MATMUL_GPU_FILE, ctx)
print(module)

Before lowering, detect what GPU this runtime actually gave us.

In [ ]:
import subprocess

def gpu_chip_and_features():
    """Auto-detect this runtime's GPU compute capability and pick a PTX
    ISA version high enough to support it (older default PTX versions
    don't support e.g. sm_89)."""
    cc = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"]
    ).decode().strip().splitlines()[0]
    major, minor = cc.split(".")
    return f"sm_{major}{minor}", "+ptx78"


GPU_CHIP, GPU_PTX_FEATURES = gpu_chip_and_features()
print("Detected GPU target:", GPU_CHIP, GPU_PTX_FEATURES)

Lower to NVVM in place, so we can see what each pipeline stage produces. `gpu-lower-to-nvvm-pipeline` compiles the kernel all the way to a real cubin (via `ptxas`, bundled with Colab's CUDA toolkit) and rewrites the host side to call into the CUDA driver at the `gpu.launch_func` site.

In [ ]:
GPU_LOWERING_PIPELINE = (
    "builtin.module("
    "func.func(llvm-request-c-wrappers),"
    f"gpu-lower-to-nvvm-pipeline{{cubin-chip={GPU_CHIP} cubin-format=bin cubin-features={GPU_PTX_FEATURES}}}"
    ")"
)

PassManager.parse(GPU_LOWERING_PIPELINE, ctx).run(module.operation)
print(module)

JIT-compile the lowered module and invoke it with numpy arrays, checking the result against `numpy`'s own matmul. Two things this needs that Part 1's CPU runs didn't:

- `shared_libs=[...]`, pointing `ExecutionEngine` at the CUDA runtime wrapper library (`mgpuModuleLoad`, `mgpuLaunchKernel`, etc.).
- `ee.initialize()` -- the compiled GPU module is loaded once via a global constructor (`llvm.mlir.global_ctors`); without explicitly running it, the module handle used inside `@matmul` is never populated and every CUDA call fails with `CUDA_ERROR_INVALID_HANDLE`.

In [ ]:
REGISTERED = []


def run_and_check(module, entry_point, args, expected, shared_libs=None, needs_init=False):
    """JIT-compile `module` (already lowered to LLVM/NVVM) and call
    `entry_point(*args)`. Each arg is a numpy array passed as a memref; by
    convention the last one is the output buffer the kernel writes into. It is
    zeroed first -- we reuse the same buffers across steps, and without this a
    kernel that writes nothing would pass on the previous step's result -- then
    checked against `expected` and returned.

    The args are copied first and the copies kept in REGISTERED for the rest
    of the session: gpu.host_register hands a buffer to the CUDA driver and
    nothing ever unregisters it, so reusing one makes the driver report
    CUDA_ERROR_HOST_MEMORY_ALREADY_REGISTERED."""
    args = [a.copy() for a in args]
    REGISTERED.extend(args)
    args[-1][...] = 0

    ee = ExecutionEngine(module, shared_libs=shared_libs or [])
    if needs_init:
        ee.initialize()
    ee.invoke(
        entry_point,
        *[ctypes.pointer(ctypes.pointer(rt.get_ranked_memref_descriptor(a)))
          for a in args],
    )

    out = args[-1]
    print("out      =", out)
    print("expected =", expected)
    np.testing.assert_allclose(out, expected)
    print("OK")
    return out


gpu_shared_libs = [
    f"{INSTALL_DIR}/lib/libmlir_cuda_runtime.so",
    f"{INSTALL_DIR}/lib/libmlir_runner_utils.so",
    f"{INSTALL_DIR}/lib/libmlir_c_runner_utils.so",
]

a = np.arange(12, dtype=np.float32).reshape(4, 3)
b = np.arange(15, dtype=np.float32).reshape(3, 5)
out = np.zeros((4, 5), dtype=np.float32)

run_and_check(module, "matmul", [a, b, out], expected=a @ b,
              shared_libs=gpu_shared_libs, needs_init=True)

## Step 2 — lowering `blas.matmul` straight to GPU

Same `session1/blas-dialect/` plugin as Part 1 (see that notebook for the full explanation), but this time you write its *GPU* lowering: a pass that builds this notebook's step-1 `gpu.launch` shape programmatically, instead of by hand.

**`blas.matmul` is already defined here.** Part 1 had you write it — the ODS in `BlasOps.td.txt` and `MatMulOp::verify()` in `BlasOps.cpp` — and this notebook clones a branch, `session1-part1-solution`, that carries that definition, so there is nothing to re-type. One edit left, and it is the GPU one.

In [ ]:
import shutil

# Colab has neither; skip the install where both are already on PATH (and
# where apt-get would need root, e.g. running locally).
if shutil.which("cmake") and shutil.which("ninja"):
    print("cmake and ninja are already installed")
else:
    !apt-get install -y -qq cmake ninja-build > /dev/null
    assert _exit_code == 0

In [ ]:
PLUGIN_SRC = f"{REPO_DIR}/session1/blas-dialect"
PLUGIN_BUILD = f"{PLUGIN_SRC}/build"
PLUGIN_SO = f"{PLUGIN_BUILD}/libBlasPlugin.so"

# Configure once, then build. Re-run this cell after every edit to a .td.txt
# or .cpp file -- those are compiled into libBlasPlugin.so, unlike the .mlir
# files, which are read fresh each time.
!cmake -S {PLUGIN_SRC} -B {PLUGIN_BUILD} -G Ninja \
    -DMLIR_DIR={INSTALL_DIR}/lib/cmake/mlir \
    -DLLVM_DIR={INSTALL_DIR}/lib/cmake/llvm \
    -DCMAKE_BUILD_TYPE=Release
assert _exit_code == 0
!ninja -C {PLUGIN_BUILD}
assert _exit_code == 0

`blas.matmul` isn't a real dialect until the plugin registers it, so we go through `mlir-opt` as a subprocess — Python's `Context` doesn't know about our plugin. With no pass pipeline, `mlir-opt` just parses and re-prints, round-tripping the op through the parser and printer your ODS definition generated.

This also confirms the clone worked. `session1/matmul_custom.mlir` is the spec the op satisfies, so it round-trips unchanged. `custom op 'blas.matmul' is unknown` here would mean the checkout is not the `session1-part1-solution` branch.

In [ ]:
MATMUL_CUSTOM_FILE = f"{REPO_DIR}/session1/matmul_custom.mlir"

!mlir-opt --load-dialect-plugin={PLUGIN_SO} {MATMUL_CUSTOM_FILE}
assert _exit_code == 0

### Write the lowering

**Open `session1/blas-dialect/lib/LowerBlasToGPU.cpp`.** One pass, `convert-blas-to-gpu`, lowers the whole dialect to `gpu.launch` kernels, with one pattern per op — the GPU counterpart of Part 1's `convert-blas-to-scf`. `MatVecToGPUPattern` is complete.

`MatMulToGPUPattern` is there with its setup already written, commented out: the operands, the `memref.cast` + `gpu.host_register` calls, the index constants, the sizes `M`, `K` and `N`, and the `eraseOp` at the end. Uncomment it, then write the part in between:

1. the `gpu::LaunchOp::create` — one block, `N` threads in x and `M` in y, the same grid as step 1's kernel;
2. its body — `row` and `col` from the block and thread ids, built the way `MatVecToGPUPattern` builds its `row`; the load of `out[row, col]`, the reduction loop over `K`, the store, and a `gpu::TerminatorOp` to end the body;

and then **register the pattern** in `runOnOperation()`. The file has a `TODO` at each spot.

**Reference: `MatVecToGPUPattern`, in that same file.** It is the same thing with a 1D thread grid: one thread per row, its `row` computed from the x ids alone. Loop bodies are built by moving the rewriter's insertion point into the block, as in Part 1, and the reduction loop, which carries a value, needs an explicit `scf::YieldOp::create`.

Forgetting to register the pattern is caught for you: the pass reports which op it failed to convert.

In [ ]:
# Opens the files below. On Colab, files.view() puts them in the editor
# pane -- use it instead of the file browser, which refuses some file
# types. Running locally there is no google.colab, so this prints the
# paths instead and you open them in your own editor.

BLAS_DIALECT = f"{REPO_DIR}/session1/blas-dialect"

try:
    from google.colab import files

    files.view(f"{BLAS_DIALECT}/lib/LowerBlasToGPU.cpp")   # pattern + registration
except ModuleNotFoundError:
    print("open in your editor:")
    print(f"    {BLAS_DIALECT}/lib/LowerBlasToGPU.cpp   (pattern + registration)")

In [ ]:
# Rebuild the plugin. Re-run this cell after EVERY edit to a .td.txt or .cpp
# file -- those are compiled into libBlasPlugin.so, unlike the .mlir files,
# which are read fresh each time. The assert matters: a failed build leaves
# the previous plugin in place, so the checks below would keep testing your
# last working version.
!ninja -C {PLUGIN_BUILD}
assert _exit_code == 0

Run your pass, writing the result to a file.

In [ ]:
LOWERED_MATMUL_FILE = "matmul_custom_lowered_gpu.mlir"

!mlir-opt --load-dialect-plugin={PLUGIN_SO} --load-pass-plugin={PLUGIN_SO} "--pass-pipeline=builtin.module(convert-blas-to-gpu)" {MATMUL_CUSTOM_FILE} -o {LOWERED_MATMUL_FILE}
assert _exit_code == 0
!cat {LOWERED_MATMUL_FILE}

Load the lowered file, run it through the same GPU pipeline as step 1, JIT it, and check the result. Compare the output above with `matmul_gpu.mlir` from step 1: the pass built the same kernel.

In [ ]:
custom_module = Module.parseFile(LOWERED_MATMUL_FILE, ctx)
PassManager.parse(GPU_LOWERING_PIPELINE, ctx).run(custom_module.operation)
run_and_check(custom_module, "matmul", [a, b, out], expected=a @ b,
              shared_libs=gpu_shared_libs, needs_init=True)